# GLADA demo: is this audio real or a deepfake?

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/JakobKauffmann/GLADA/blob/main/notebooks/00_demo/glada_demo.ipynb)

This notebook installs the `glada` package, downloads a trained detector from
the ICTAI 2026 paper and MS thesis (per-group VIB on the lowest layer group,
WavLM-Base-Plus, seed 42), and uses it to label audio clips as real speech or a
deepfake.

Runs on a free Colab runtime (CPU is fine; GPU is faster). The first run
downloads about 850 MB: the GLADA weights and the WavLM-Base-Plus backbone.

## 1. Install

In [ ]:
!pip install -q "git+https://github.com/JakobKauffmann/GLADA.git#subdirectory=glada" safetensors

## 2. Load the trained model

The release has two files: `model.safetensors` (weights) and `config.json`
(the training configuration). The model card is on the
[release page](https://github.com/JakobKauffmann/GLADA/releases/tag/v1.0-weights).

In [ ]:
import json, urllib.request
import numpy as np
import torch
import torch.nn.functional as F
import librosa
from safetensors.torch import load_file

from glada import GLADAConfig
from glada.models.glada_model import PerGroupVIBModel

RELEASE = "https://github.com/JakobKauffmann/GLADA/releases/download/v1.0-weights"
for name in ["config.json", "model.safetensors"]:
    urllib.request.urlretrieve(f"{RELEASE}/{name}", name)

meta = json.load(open("config.json"))
config = GLADAConfig(**meta["glada_config"])
device = "cuda" if torch.cuda.is_available() else "cpu"

model = PerGroupVIBModel(config, vib_disabled_groups=tuple(meta["vib_disabled_groups"]))
model.load_state_dict(load_file("model.safetensors"))
model = model.to(device).eval()
print(f"Loaded on {device}. In-the-Wild EER at this checkpoint: {meta['itw_eer_percent']}%")

## 3. A prediction helper

`predict` loads a clip at 16 kHz, crops or pads it to the 4 seconds the model
was trained on, and prints the probability of real speech and of a deepfake.

In [ ]:
def predict(path, threshold=0.5):
    wav, _ = librosa.load(path, sr=16000, mono=True)
    n = config.max_audio_len
    wav = wav[:n] if len(wav) >= n else np.pad(wav, (0, n - len(wav)))
    x = torch.from_numpy(wav).float().unsqueeze(0).to(device)
    with torch.no_grad():
        probs = F.softmax(model(x)["spoof_logits"], dim=1)[0].cpu().numpy()
    p_fake, p_real = float(probs[0]), float(probs[1])

    label = "REAL" if p_real >= threshold else "DEEPFAKE"
    bar = lambda p: "#" * round(p * 30) + "-" * (30 - round(p * 30))
    print(f"Clip:        {path.split('/')[-1]}")
    print(f"Prediction:  {label}")
    print(f"  Real       {bar(p_real)}  {p_real:6.1%}")
    print(f"  Deepfake   {bar(p_fake)}  {p_fake:6.1%}\n")
    return {"label": label, "real": p_real, "deepfake": p_fake}

## 4. Real speech

A LibriSpeech audiobook recording that ships with librosa. It is a real
person reading aloud.

In [ ]:
real = predict(librosa.ex("libri1"))

## 5. Synthetic speech

This generates a clip with espeak-ng, an open-source text-to-speech engine,
so we know it is fake.

In [ ]:
!apt-get -qq install -y espeak-ng > /dev/null
!espeak-ng -w synthetic.wav "This sentence was generated by a computer voice, not spoken by a person."
fake = predict("synthetic.wav")

## 6. Try your own audio

Upload a clip (wav, mp3, flac, or ogg). The model looks at the first 4 seconds.

In [ ]:
from google.colab import files
for name in files.upload():
    predict(name)

## Limits

The detector was trained on English speech from ASVspoof 2019 LA, with
In-the-Wild audio as an unlabelled target domain. It can miss deepfakes from
generators it has not seen. Cross-generator
generalization is the open problem the paper studies. Treat these predictions
as a research demo, not a verdict on real content.

**More:** training notebooks are in `notebooks/04_pgvib_development/`,
cross-dataset results in `notebooks/05_cross_dataset_eval/`, and the thesis
PDF in `docs/`.